# 5. Monte Carlo Methods

**Kaynak:** Sutton & Barto, *Reinforcement Learning: An Introduction*, 2nd Edition (2018)
- **Bölüm 5: Monte Carlo Methods** (Sayfa 91-113)

## İçindekiler
1. MC'ye Giriş *(s. 91-92)*
2. MC Prediction *(s. 92-96)*
3. MC Control *(s. 96-103)*
4. On-policy vs Off-policy *(s. 99-103)*
5. Importance Sampling *(s. 103-108)*
6. Incremental Implementation *(s. 110-111)*

---
## 5.1 Monte Carlo Nedir?

📖 **Referans:** Sutton & Barto, Sayfa 91-92

> *"Monte Carlo methods require only experience—sample sequences of states, actions, and rewards from actual or simulated interaction with an environment."* (s. 91)

**Monte Carlo (MC)** metodları, **deneyimden** (experience) öğrenir - model gerektirmez!

### DP vs MC (s. 91)

| Özellik | DP | MC |
|---------|----|----|  
| Model | Gerekli | Gerekli değil |
| Update | Her state için | Episode sonunda |
| Bootstrap | Evet | Hayır |

> *"Monte Carlo methods are ways of solving the reinforcement learning problem based on averaging sample returns."* (s. 91)

### MC'nin Temel Fikri (s. 92)

Value'yu **sample returns**'ların ortalaması olarak tahmin et:

$$v_\pi(s) \approx \text{average of returns observed after visits to } s$$

> *"As more returns are observed, the average should converge to the expected value."* (s. 92)

In [ ]:
# Kod Örneği: Blackjack Environment
# Referans: Sutton & Barto, Example 5.1 (s. 93-94) - Blackjack

import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from typing import List, Tuple, Dict

class BlackjackEnv:
    """
    Simplified Blackjack environment.
    Referans: Example 5.1 (s. 93-94)
    
    "The object of the popular casino card game of blackjack is to obtain cards 
    the sum of whose numerical values is as great as possible without exceeding 21."
    
    State: (player_sum, dealer_showing, usable_ace)
    Actions: 0 = stick, 1 = hit
    """
    
    def __init__(self):
        self.action_space = [0, 1]  # stick, hit (s. 93)
    
    def draw_card(self):
        """
        Draw a card (1-10, face cards = 10).
        "Face cards (Jack, Queen, King) count as 10" (s. 93)
        """
        card = min(np.random.randint(1, 14), 10)
        return card
    
    def draw_hand(self):
        """Draw initial hand."""
        return [self.draw_card(), self.draw_card()]
    
    def usable_ace(self, hand):
        """
        Check if hand has usable ace.
        "An ace can count either as 1 or as 11" (s. 93)
        """
        return 1 in hand and sum(hand) + 10 <= 21
    
    def sum_hand(self, hand):
        """Sum of hand, ace = 11 if usable."""
        if self.usable_ace(hand):
            return sum(hand) + 10
        return sum(hand)
    
    def is_bust(self, hand):
        """Check if hand busts (> 21)."""
        return self.sum_hand(hand) > 21
    
    def reset(self):
        """Start new episode."""
        self.player = self.draw_hand()
        self.dealer = self.draw_hand()
        
        # "Player starts with sum of 12-21" (s. 93)
        while self.sum_hand(self.player) < 12:
            self.player.append(self.draw_card())
        
        return self._get_state()
    
    def _get_state(self):
        return (
            self.sum_hand(self.player),
            self.dealer[0],  # "Dealer showing" (s. 93)
            self.usable_ace(self.player)
        )
    
    def step(self, action):
        """
        Take action, return (state, reward, done).
        "The rewards of +1, −1, and 0 are given for winning, losing, 
        and drawing" (s. 93)
        """
        if action == 1:  # Hit - "requests additional cards"
            self.player.append(self.draw_card())
            
            if self.is_bust(self.player):
                return self._get_state(), -1, True  # "goes bust" (s. 93)
            else:
                return self._get_state(), 0, False
        
        else:  # Stick - "stops with current sum"
            # "Dealer's strategy: sticks on any sum of 17 or greater" (s. 93)
            while self.sum_hand(self.dealer) < 17:
                self.dealer.append(self.draw_card())
            
            player_sum = self.sum_hand(self.player)
            dealer_sum = self.sum_hand(self.dealer)
            
            if self.is_bust(self.dealer) or player_sum > dealer_sum:
                reward = 1  # Win
            elif player_sum < dealer_sum:
                reward = -1  # Lose
            else:
                reward = 0  # Draw
            
            return self._get_state(), reward, True

# Test
env = BlackjackEnv()
state = env.reset()
print(f"Initial state: {state}")
print(f"(player_sum={state[0]}, dealer_showing={state[1]}, usable_ace={state[2]})")

---
## 5.2 MC Prediction (Policy Evaluation)

📖 **Referans:** Sutton & Barto, Sayfa 92-96, Section 5.1

> *"We begin by considering Monte Carlo methods for learning the state-value function for a given policy."* (s. 92)

Verilen bir policy $\pi$ için $V^\pi$ veya $Q^\pi$ tahmin et.

### First-Visit MC vs Every-Visit MC (s. 92)

> *"The first-visit MC method estimates $v_\pi(s)$ as the average of the returns following first visits to s, whereas the every-visit MC method averages the returns following all visits to s."*

### First-Visit MC Prediction Algorithm (s. 92)

```
Initialize:
    V(s) ∈ R, arbitrarily, for all s ∈ S
    Returns(s) ← an empty list, for all s ∈ S

Loop forever (for each episode):
    Generate an episode following π
    G ← 0
    Loop for each step of episode, t = T-1, T-2, ..., 0:
        G ← γG + R_{t+1}
        Unless S_t appears in S_0, S_1, ..., S_{t-1}:
            Append G to Returns(S_t)
            V(S_t) ← average(Returns(S_t))
```

> *"Both first-visit MC and every-visit MC converge to $v_\pi(s)$ as the number of visits (or first visits) to s goes to infinity."* (s. 92)

In [ ]:
def generate_episode(env, policy):
    """
    Policy'yi takip ederek bir episode oluştur.
    Referans: "Generate an episode following π" (s. 92)
    
    Returns:
        episode: List of (state, action, reward)
    """
    episode = []
    state = env.reset()
    
    while True:
        action = policy(state)
        next_state, reward, done = env.step(action)
        episode.append((state, action, reward))
        
        if done:
            break
        state = next_state
    
    return episode

# Simple policy from Example 5.1 (s. 93)
# "The player sticks if his sum is 20 or 21, and otherwise hits"
def simple_policy(state):
    player_sum, dealer_showing, usable_ace = state
    return 0 if player_sum >= 20 else 1

# Test
episode = generate_episode(env, simple_policy)
print("Episode generated following simple policy:")
for s, a, r in episode:
    print(f"  State: {s}, Action: {'stick' if a==0 else 'hit'}, Reward: {r}")

In [ ]:
def mc_prediction_v(env, policy, n_episodes=10000, gamma=1.0):
    """
    First-Visit MC Prediction for V(s).
    Referans: Algorithm (s. 92)
    
    "Loop for each step of episode, t = T-1, T-2, ..., 0:
        G ← γG + R_{t+1}
        Unless S_t appears in S_0, S_1, ..., S_{t-1}:
            Append G to Returns(S_t)
            V(S_t) ← average(Returns(S_t))"
    """
    # "V(s) ∈ R, arbitrarily, for all s ∈ S"
    V = defaultdict(float)
    # "Returns(s) ← an empty list, for all s ∈ S"
    returns = defaultdict(list)
    
    for _ in range(n_episodes):
        episode = generate_episode(env, policy)
        
        # Calculate returns - "G ← 0"
        G = 0
        visited_states = set()
        
        # "Loop for each step of episode, t = T-1, T-2, ..., 0"
        for t in range(len(episode) - 1, -1, -1):
            state, action, reward = episode[t]
            G = gamma * G + reward  # "G ← γG + R_{t+1}"
            
            # "Unless S_t appears in S_0, S_1, ..., S_{t-1}" (First-visit check)
            if state not in visited_states:
                visited_states.add(state)
                returns[state].append(G)  # "Append G to Returns(S_t)"
                V[state] = np.mean(returns[state])  # "V(S_t) ← average(Returns(S_t))"
    
    return V, returns

V, returns = mc_prediction_v(env, simple_policy, n_episodes=50000)
print(f"First-visit MC estimated V for {len(V)} states")

In [ ]:
def plot_value_function(V, title="Value Function"):
    """
    Blackjack value function'ı görselleştir.
    Referans: Figure 5.1 (s. 94) - "Approximate state-value functions 
    for the blackjack policy that sticks only on 20 or 21"
    """
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    for idx, usable_ace in enumerate([False, True]):
        ax = axes[idx]
        
        # "player's current sum (12–21)" (s. 94)
        player_range = range(12, 22)
        # "dealer's showing card (ace–10)" (s. 94)
        dealer_range = range(1, 11)
        
        grid = np.zeros((len(player_range), len(dealer_range)))
        
        for i, player in enumerate(player_range):
            for j, dealer in enumerate(dealer_range):
                state = (player, dealer, usable_ace)
                grid[i, j] = V.get(state, 0)
        
        im = ax.imshow(grid, cmap='RdYlGn', origin='lower', 
                       extent=[0.5, 10.5, 11.5, 21.5], aspect='auto',
                       vmin=-1, vmax=1)
        
        ax.set_xlabel('Dealer Showing')
        ax.set_ylabel('Player Sum')
        ax.set_title(f'Usable Ace: {usable_ace}')
        plt.colorbar(im, ax=ax)
    
    plt.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()

plot_value_function(V, "MC Prediction: V(s) - Figure 5.1 (s. 94)")

---
## 5.3 MC Control

📖 **Referans:** Sutton & Barto, Sayfa 96-103, Section 5.3-5.4

> *"The overall idea is to proceed according to the same pattern as for DP: policy evaluation followed by policy improvement."* (s. 96)

Optimal policy bulmak için MC kullan.

### MC with Exploring Starts (s. 99)

> *"We made two unlikely assumptions above in order to easily obtain this guarantee of convergence for the Monte Carlo method. One was that the episodes have exploring starts"* (s. 99)

Her state-action çiftiyle başlama şansı olsun.

### On-policy MC Control: ε-soft Policies (s. 100-101)

> *"In on-policy control methods the policy is generally soft, meaning that π(a|s) > 0 for all s ∈ S and all a ∈ A(s)"* (s. 100)

**ε-greedy** policy kullan:

$$\pi(a|s) = \begin{cases} 1 - \epsilon + \frac{\epsilon}{|A(s)|} & \text{if } a = \arg\max_a Q(s,a) \\ \frac{\epsilon}{|A(s)|} & \text{otherwise} \end{cases}$$

In [ ]:
def mc_control_epsilon_greedy(env, n_episodes=100000, gamma=1.0, epsilon=0.1):
    """
    On-policy MC Control with ε-greedy policy.
    Referans: Algorithm on page 101
    
    "On-policy first-visit MC control (for ε-soft policies), estimates π ≈ π*"
    """
    # "Initialize, for all s ∈ S, a ∈ A(s):"
    # "Q(s, a) ∈ R (arbitrarily)"
    Q = defaultdict(lambda: np.zeros(2))
    # "Returns(s,a) ← empty list"
    returns = defaultdict(list)
    
    def epsilon_greedy_policy(state):
        """
        ε-greedy policy (s. 100)
        "With probability ε select an action at random"
        """
        if np.random.random() < epsilon:
            return np.random.randint(2)
        else:
            return np.argmax(Q[state])
    
    for ep in range(n_episodes):
        # "Generate an episode using π"
        episode = generate_episode(env, epsilon_greedy_policy)
        
        # "G ← 0"
        G = 0
        visited_sa = set()
        
        # "Loop for each step of episode, t = T−1, T−2, . . . , 0:"
        for t in range(len(episode) - 1, -1, -1):
            state, action, reward = episode[t]
            G = gamma * G + reward  # "G ← γG + R_{t+1}"
            
            sa = (state, action)
            # "Unless the pair S_t, A_t appears in (S_0,A_0), (S_1,A_1)..."
            if sa not in visited_sa:
                visited_sa.add(sa)
                returns[sa].append(G)  # "Append G to Returns(S_t, A_t)"
                Q[state][action] = np.mean(returns[sa])  # "Q(S_t, A_t) ← average(Returns(S_t, A_t))"
    
    # Extract greedy policy: "π(s) ← argmax_a Q(s,a)"
    policy = {}
    for state in Q:
        policy[state] = np.argmax(Q[state])
    
    return Q, policy

Q, optimal_policy = mc_control_epsilon_greedy(env, n_episodes=100000)
print(f"On-policy MC Control learned Q for {len(Q)} states")

In [ ]:
def plot_policy(policy, title="Policy"):
    """
    Blackjack policy'sini görselleştir.
    Referans: Figure 5.2 (s. 95) - "Optimal policy and state-value function 
    for blackjack, found by Monte Carlo ES"
    """
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    for idx, usable_ace in enumerate([False, True]):
        ax = axes[idx]
        
        player_range = range(12, 22)
        dealer_range = range(1, 11)
        
        grid = np.zeros((len(player_range), len(dealer_range)))
        
        for i, player in enumerate(player_range):
            for j, dealer in enumerate(dealer_range):
                state = (player, dealer, usable_ace)
                grid[i, j] = policy.get(state, 1)  # Default: hit
        
        im = ax.imshow(grid, cmap='RdYlGn', origin='lower',
                       extent=[0.5, 10.5, 11.5, 21.5], aspect='auto',
                       vmin=0, vmax=1)
        
        ax.set_xlabel('Dealer Showing')
        ax.set_ylabel('Player Sum')
        ax.set_title(f'Usable Ace: {usable_ace}')
        
        # Legend
        from matplotlib.patches import Patch
        legend_elements = [Patch(facecolor='red', label='Stick (0)'),
                          Patch(facecolor='green', label='Hit (1)')]
        ax.legend(handles=legend_elements, loc='upper right')
    
    plt.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()

plot_policy(optimal_policy, "MC Control: Optimal Policy - Figure 5.2 (s. 95)")

---
## 5.4 On-policy vs Off-policy

📖 **Referans:** Sutton & Barto, Sayfa 99-103, Section 5.4-5.5

### On-policy (s. 100)
> *"In on-policy methods we attempt to evaluate or improve the policy that is used to make decisions."*

- **Aynı** policy'yi hem öğrenmek hem de veri toplamak için kullan
- Öğrenilen policy = behavior policy
- Örnek: ε-greedy MC Control

### Off-policy (s. 103)
> *"Off-policy methods... learn the value function of a target policy from data generated by a different behavior policy."*

- **Farklı** policy'ler: target policy (öğrenilen) vs behavior policy (veri toplayan)
- Daha esnek, ama daha karmaşık
- **Importance Sampling** gerektirir

> *"Off-policy learning is also seen as key to learning multi-step predictive models of the world's dynamics"* (s. 103)

---
## 5.5 Importance Sampling

📖 **Referans:** Sutton & Barto, Sayfa 103-108, Section 5.5

> *"Almost all off-policy methods utilize importance sampling, a general technique for estimating expected values under one distribution given samples from another."* (s. 103)

Bir dağılımdan sample alıp başka bir dağılımın expected value'sunu tahmin etme tekniği.

### Importance Sampling Ratio - Equation 5.3 (s. 104)

$$\rho_{t:T-1} = \prod_{k=t}^{T-1} \frac{\pi(A_k|S_k)}{b(A_k|S_k)}$$

Burada:
- $\pi$: Target policy
- $b$: Behavior policy

### Ordinary Importance Sampling - Equation 5.4 (s. 104)

$$V(s) = \frac{\sum_{t \in \mathcal{T}(s)} \rho_{t:T(t)-1} G_t}{|\mathcal{T}(s)|}$$

### Weighted Importance Sampling - Equation 5.5 (s. 105)

$$V(s) = \frac{\sum_{t \in \mathcal{T}(s)} \rho_{t:T(t)-1} G_t}{\sum_{t \in \mathcal{T}(s)} \rho_{t:T(t)-1}}$$

> *"In practice, the weighted estimator usually has dramatically lower variance and is strongly preferred."* (s. 105)

In [ ]:
def mc_off_policy_prediction(env, target_policy, n_episodes=10000, gamma=1.0):
    """
    Off-policy MC Prediction using Weighted Importance Sampling.
    Referans: Algorithm (s. 110)
    
    "Off-policy MC prediction (policy evaluation) for estimating Q ≈ q_π,
    using weighted importance sampling"
    
    Behavior policy: random (covers all actions)
    """
    # "Initialize, for all s ∈ S, a ∈ A(s):"
    Q = defaultdict(lambda: np.zeros(2))
    C = defaultdict(lambda: np.zeros(2))  # "C(s,a) ← 0" Cumulative weights
    
    def behavior_policy(state):
        """
        Random behavior policy.
        "b is any policy with coverage of π" (s. 103)
        """
        return np.random.randint(2)
    
    for _ in range(n_episodes):
        # "Generate an episode following b"
        episode = generate_episode(env, behavior_policy)
        
        G = 0  # "G ← 0"
        W = 1.0  # "W ← 1"
        
        # "Loop for each step of episode, t = T−1, T−2, . . . , 0, while W ≠ 0:"
        for t in range(len(episode) - 1, -1, -1):
            state, action, reward = episode[t]
            G = gamma * G + reward  # "G ← γG + R_{t+1}"
            
            # "C(S_t,A_t) ← C(S_t,A_t) + W"
            C[state][action] += W
            # "Q(S_t,A_t) ← Q(S_t,A_t) + (W/C(S_t,A_t))[G − Q(S_t,A_t)]"
            Q[state][action] += (W / C[state][action]) * (G - Q[state][action])
            
            # Target policy action
            target_action = target_policy(state)
            
            # "If A_t ≠ π(S_t) then exit inner loop"
            if action != target_action:
                break
            
            # "W ← W · (1/b(A_t|S_t))"
            # π is deterministic (1 for target action), b is random (0.5)
            W *= 1.0 / 0.5
    
    return Q

# Target: stick on 20 or 21 (Example 5.1 policy)
def target_policy(state):
    return 0 if state[0] >= 20 else 1

Q_off = mc_off_policy_prediction(env, target_policy, n_episodes=50000)
print(f"Off-policy MC with Weighted IS estimated Q for {len(Q_off)} states")

In [ ]:
# Variance comparison: Ordinary vs Weighted IS
def compare_importance_sampling(env, n_runs=100, episode_counts=[1, 10, 100, 1000, 10000]):
    """
    Compare ordinary and weighted IS estimates.
    """
    # Single state to estimate
    target_state = (13, 2, True)  # Example state
    
    def target_policy(state):
        return 0 if state[0] >= 20 else 1
    
    def behavior_policy(state):
        return np.random.randint(2)
    
    results = {'ordinary': [], 'weighted': []}
    
    for n_episodes in episode_counts:
        ordinary_estimates = []
        weighted_estimates = []
        
        for run in range(n_runs):
            # Collect episodes starting from target_state
            ordinary_returns = []
            weighted_num = 0
            weighted_denom = 0
            
            for _ in range(n_episodes):
                # Force start from target state (simplified)
                env.reset()
                env.player = [3, 10] if target_state[2] else [3, 10]  # Simplified
                
                episode = generate_episode(env, behavior_policy)
                
                if len(episode) > 0:
                    G = sum([r for _, _, r in episode])
                    
                    # Calculate importance ratio
                    rho = 1.0
                    for state, action, _ in episode:
                        target_action = target_policy(state)
                        if action == target_action:
                            rho *= 2.0  # π/b = 1/0.5
                        else:
                            rho = 0
                            break
                    
                    ordinary_returns.append(rho * G)
                    weighted_num += rho * G
                    weighted_denom += rho
            
            if ordinary_returns:
                ordinary_estimates.append(np.mean(ordinary_returns))
            if weighted_denom > 0:
                weighted_estimates.append(weighted_num / weighted_denom)
        
        if ordinary_estimates:
            results['ordinary'].append(np.var(ordinary_estimates))
        if weighted_estimates:
            results['weighted'].append(np.var(weighted_estimates))
    
    return results, episode_counts

print("Importance Sampling comparison (simplified demo)")

---
## 5.6 Incremental Implementation

📖 **Referans:** Sutton & Barto, Sayfa 110-111, Section 5.6

> *"Monte Carlo prediction methods can be implemented incrementally, on an episode-by-episode basis"* (s. 110)

Her episode'dan sonra tüm return'leri saklamak yerine **incremental** güncelleme:

### Incremental Mean Update (s. 110)

$$V(S_t) \leftarrow V(S_t) + \alpha [G_t - V(S_t)]$$

Bu, sample average'ın bir genellemesidir:
- $\alpha = 1/n$: Sample average (non-stationary değil)
- $\alpha$ sabit: Non-stationary problemler için

> *"For nonstationary problems, it may be desirable to use a constant step-size parameter"* (s. 110)

In [ ]:
def mc_control_incremental(env, n_episodes=100000, gamma=1.0, alpha=0.01, epsilon=0.1):
    """
    MC Control with constant-α (incremental updates).
    Referans: "For nonstationary problems, it may be desirable to use 
    a constant step-size parameter" (s. 110)
    
    Uses update: Q(S_t,A_t) ← Q(S_t,A_t) + α[G_t − Q(S_t,A_t)]
    """
    Q = defaultdict(lambda: np.zeros(2))
    
    def epsilon_greedy(state):
        if np.random.random() < epsilon:
            return np.random.randint(2)
        return np.argmax(Q[state])
    
    rewards_per_episode = []
    
    for ep in range(n_episodes):
        episode = generate_episode(env, epsilon_greedy)
        
        episode_reward = sum([r for _, _, r in episode])
        rewards_per_episode.append(episode_reward)
        
        # "G ← 0"
        G = 0
        for t in range(len(episode) - 1, -1, -1):
            state, action, reward = episode[t]
            G = gamma * G + reward  # "G ← γG + R_{t+1}"
            
            # "Q(S_t,A_t) ← Q(S_t,A_t) + α[G_t − Q(S_t,A_t)]"
            Q[state][action] += alpha * (G - Q[state][action])
    
    return Q, rewards_per_episode

Q_inc, rewards = mc_control_incremental(env, n_episodes=100000)

# Learning curve
window = 1000
smoothed = np.convolve(rewards, np.ones(window)/window, mode='valid')

plt.figure(figsize=(10, 5))
plt.plot(smoothed)
plt.xlabel('Episode')
plt.ylabel('Average Reward (per 1000 episodes)')
plt.title('MC Control Learning Curve (constant-α, s. 110)')
plt.grid(True, alpha=0.3)
plt.show()

---
## Özet

📖 **Chapter 5 Key Points (s. 91-113)**

| Kavram | Sayfa | Açıklama |
|--------|-------|----------|
| **MC** | s. 91 | Episode sonunda öğrenme, model-free |
| **First-Visit MC** | s. 92 | İlk ziyareti say |
| **Every-Visit MC** | s. 92 | Tüm ziyaretleri say |
| **MC Control** | s. 96-103 | Policy optimization (ε-greedy) |
| **On-policy** | s. 100 | Behavior = Target |
| **Off-policy** | s. 103 | Behavior ≠ Target, IS gerekli |
| **Importance Sampling** | s. 103-108 | Ratio: $\rho = \pi(a|s)/b(a|s)$ |

### MC'nin Avantajları (s. 91-92)
- Model gerektirmez
- Basit ve anlaşılır
- Episode-based tasks için uygun

### MC'nin Dezavantajları (s. 91)
- Episode **bitmeli** (continuing tasks için uygun değil)
- Yüksek variance (tüm episode boyunca)

### Anahtar Denklemler

**Importance Sampling Ratio** (Eq. 5.3, s. 104):
$$\rho_{t:T-1} = \prod_{k=t}^{T-1} \frac{\pi(A_k|S_k)}{b(A_k|S_k)}$$

**Weighted IS** (Eq. 5.5, s. 105):
$$V(s) = \frac{\sum_{t \in \mathcal{T}(s)} \rho_{t:T(t)-1} G_t}{\sum_{t \in \mathcal{T}(s)} \rho_{t:T(t)-1}}$$

---
### Sonraki Notebook
**06 - Temporal Difference Learning** *(Chapter 6, s. 115-145)*: TD(0), SARSA, Q-Learning